In [1]:
import pandas as pd 
import numpy as np
from s3_utils import read_s3_csv, upload_to_s3, get_s3_client
import re

In [2]:
s3 = get_s3_client()
BUCKET = "projet-accidents-jedha"

# 1. On liste TOUS les fichiers du dossier bronze
response = s3.list_objects_v2(Bucket=BUCKET, Prefix='bronze/')

# 2. On crée une liste vide pour stocker nos DataFrames
dfs_to_merge = []

# 3. La boucle magique intelligente
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    # On filtre pour ne garder que la catégorie "véhicules"
    if "vehicules" in file_key.lower():
        
        # --- LOGIQUE D'EXTRACTION DE L'ANNÉE ---
        # On cherche 4 chiffres dans le nom (ex: 2024)
        match = re.search(r'(\d{4})', file_key)
        
        if match:
            annee = int(match.group(1))
            
            # On définit le séparateur selon l'année
            if annee >= 2024:
                sep = ','
            else:
                sep = ';'
            
            print(f"🔎 Trouvé : {file_key} | Année: {annee} | Séparateur: '{sep}'")
            
            # On lit le fichier avec le bon séparateur
            df_temp = read_s3_csv(file_key, separator=sep)
            dfs_to_merge.append(df_temp)

# 4. On fusionne tout d'un coup
if dfs_to_merge:
    df_vehicule = pd.concat(dfs_to_merge, ignore_index=True)
    print(f"✅ Terminé ! {len(dfs_to_merge)} fichiers fusionnés.")
else:
    print("⚠️ Aucun fichier trouvé, vérifie le mot-clé ou le dossier.")

🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - vehicules 2021.csv | Année: 2021 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - vehicules 2022.csv | Année: 2022 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - vehicules 2023.csv | Année: 2023 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - vehicules 2024.csv | Année: 2024 | Séparateur: ','
✅ Terminé ! 4 fichiers fusionnés.


In [3]:
# 1. Liste des colonnes à supprimer
colonnes_a_drop = ['senc', 'choc', 'manv', 'occutc']

# 2. On les retire du DataFrame
# errors='ignore' permet d'éviter un plantage si une colonne est déjà absente
df_vehicule = df_vehicule.drop(columns=colonnes_a_drop, errors='ignore')

# 3. Vérification : on affiche les colonnes restantes
print("Colonnes restantes dans le fichier :")
print(df_vehicule.columns.tolist())



Colonnes restantes dans le fichier :
['Num_Acc', 'id_vehicule', 'num_veh', 'catv', 'obs', 'obsm', 'motor']


In [4]:
df_vehicule["id_vehicule"] = (
    df_vehicule["id_vehicule"]
    .astype(str)                  # s'assurer que c'est du texte
    .str.replace(r"\s+", "", regex=True)  # enlever tous les espaces
    .astype(int)                  # convertir en entier
)

In [5]:
# On utilise ta fonction d'upload
# 'vehicules_cleaned.csv' est le nom que le fichier aura sur S3
upload_to_s3(df_vehicule, "vehicules_cleaned.csv", folder="silver")

✅ Mission accomplie : silver/vehicules_cleaned.csv est sur S3 !
